# Bangalore House Prices: Data Cleaning Pipeline

This notebook is an automated data pipeline designed to clean the raw **Bangalore House Prices** dataset. It uses the `kaggle_toolbox` to optimize memory and handle data quality issues, then outputs a cleaned CSV for downstream modeling.

### **Resources:**
 - **Utility Script**: [kaggle-toolbox](https://www.kaggle.com/code/ameythakur20/kaggle-toolbox)
 - **Dataset**: [bangalore-house-prices](https://www.kaggle.com/datasets/ameythakur20/bangalore-house-prices)

**Author**: Amey Thakur (https://www.kaggle.com/ameythakur20)

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Locate the utility script in the Kaggle filesystem
UTILITY_PATH = "/kaggle/usr/lib/ameythakur20/kaggle-toolbox"
if os.path.exists(UTILITY_PATH):
    sys.path.append(UTILITY_PATH)

import kaggle_toolbox as tb

In [ ]:
def clean_sqft(value):
    if isinstance(value, float) or isinstance(value, int):
        return float(value)
    try:
        val_str = str(value)
        if '-' in val_str:
            parts = val_str.split('-')
            return (float(parts[0]) + float(parts[1])) / 2
        return float(val_str)
    except:
        return np.nan

# 1. Locate and Load the Dataset
input_files = tb.find_input("bengaluru_house_prices")
DATA_PATH = input_files[0] if input_files else "bengaluru_house_prices.csv"

with tb.timer("Data Initialization"):
    df = pd.read_csv(DATA_PATH)

# 2. Clean 'total_sqft' with range-to-average conversion
df['total_sqft_cleaned'] = df['total_sqft'].apply(clean_sqft)
df = df.dropna(subset=['total_sqft_cleaned', 'bath', 'price'])

# 3. Optimize Memory
df = tb.reduce_mem_usage(df)

# 4. Final Data Clean: Remove useless columns
useless = tb.find_useless_columns(df)
cols_to_drop = useless["constant"] + [p[0] for p in useless["duplicate"]]
if cols_to_drop:
    df.drop(columns=cols_to_drop, inplace=True)

# 5. Save the Cleaned Pipeline Output
df.to_csv("bengaluru_house_prices_cleaned.csv", index=False)
print(f"Pipeline complete. Cleaned file saved with {len(df)} rows.")